# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
# Notebook-local constants and metadata
TASK_ID = 'task378'
MODEL_VERSION = 'task378-symbolic-frame-corner-rays-v1'
FAMILY = 'arc_static_symbolic'
SUBTYPE = 'dense payload in sparse frame, far closed frame-corner diagonal rays'

In [2]:
# ONNX dependency setup.
import importlib.util, subprocess, sys
required = {'onnx':'onnx', 'onnxruntime':'onnxruntime', 'sklearn':'scikit-learn', 'torch':'torch'}
missing = [pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import onnx, onnxruntime as ort
print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.5 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [3]:

# Shared symbolic ARC -> static ONNX builder using explicit tensor operations.
import json, os, zipfile, math, subprocess, sys
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort

H = W = 30
CH = 10
FORBIDDEN = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}
torch.set_num_threads(2)


def find_task_json(task_id):
    candidates = [
        Path.cwd() / f'{task_id}.json',
        Path.cwd() / f'{task_id.replace("task", "task")}.json',
        Path('/mnt/data') / f'{task_id}.json',
        Path('/kaggle/working') / f'{task_id}.json',
    ]
    for p in candidates:
        if p.exists():
            return p
    for base in [Path('/kaggle/input'), Path.cwd(), Path('/kaggle/working')]:
        if base.exists():
            hits = list(base.rglob(f'{task_id}.json'))
            if hits:
                return hits[0]
    raise FileNotFoundError(f'Could not locate {task_id}.json')


def load_task(task_id):
    path = find_task_json(task_id)
    with open(path) as f:
        return json.load(f), path


def examples_for_scope(task, scope='all'):
    if scope == 'visible':
        return task.get('train', []) + task.get('test', [])
    if scope == 'train':
        return task.get('train', [])
    if scope == 'test':
        return task.get('test', [])
    return task.get('train', []) + task.get('test', []) + task.get('arc-gen', [])


def grid_to_tensor(grid):
    arr = np.zeros((1, CH, H, W), np.float32)
    for r, row in enumerate(grid[:H]):
        for c, v in enumerate(row[:W]):
            arr[0, int(v), r, c] = 1.0
    return arr


def expected_tensor(grid):
    return grid_to_tensor(grid)


class CoordModule(nn.Module):
    def __init__(self):
        super().__init__()
        rr = torch.arange(H, dtype=torch.float32).view(1, 1, H, 1).expand(1, 1, H, W)
        cc = torch.arange(W, dtype=torch.float32).view(1, 1, 1, W).expand(1, 1, H, W)
        self.register_buffer('rr', rr)
        self.register_buffer('cc', cc)
        self.big = 1000.0


def shift_fixed(t, dr, dc):
    # Move source cells by fixed (dr, dc), dropping anything beyond the static 30x30 canvas.
    ys = max(0, -dr)
    ye = H - max(0, dr)
    xs = max(0, -dc)
    xe = W - max(0, dc)
    src = t[:, :, ys:ye, xs:xe]
    top = ys + dr
    left = xs + dc
    bottom = H - top - (ye - ys)
    right = W - left - (xe - xs)
    return F.pad(src, (left, right, top, bottom))


def ray_from_seed(seed, dr, dc):
    ray = torch.zeros_like(seed)
    cur = seed
    for _ in range(29):
        cur = shift_fixed(cur, dr, dc)
        ray = ray + cur
    return (ray > 0.0).float()


class Task168DiagonalLCompletion(CoordModule):
    """Find every 2x2 L-triomino and extend the diagonal from its missing corner outward."""
    def __init__(self):
        super().__init__()

    def ray_from_mask(self, m):
        m00 = m[:, :, 0:H-1, 0:W-1]
        m01 = m[:, :, 0:H-1, 1:W]
        m10 = m[:, :, 1:H, 0:W-1]
        m11 = m[:, :, 1:H, 1:W]
        seed_tl = (1.0 - m00) * m01 * m10 * m11
        seed_tr = m00 * (1.0 - m01) * m10 * m11
        seed_bl = m00 * m01 * (1.0 - m10) * m11
        seed_br = m00 * m01 * m10 * (1.0 - m11)
        seed_tl = F.pad(seed_tl, (0, 1, 0, 1))
        seed_tr = F.pad(seed_tr, (1, 0, 0, 1))
        seed_bl = F.pad(seed_bl, (0, 1, 1, 0))
        seed_br = F.pad(seed_br, (1, 0, 1, 0))
        ray = (ray_from_seed(seed_tl, -1, -1) +
               ray_from_seed(seed_tr, -1,  1) +
               ray_from_seed(seed_bl,  1, -1) +
               ray_from_seed(seed_br,  1,  1))
        return (ray > 0.0).float()

    def forward(self, x):
        bg = x[:, 0:1]
        colored = x[:, 1:CH]
        # This task has exactly one non-background colour. Compute the geometry once
        # on the union mask, then route the additions back to the present colour channel.
        m = (colored.sum(dim=1, keepdim=True) > 0.0).float()
        present = (colored.sum(dim=(2, 3), keepdim=True) > 0.0).float()
        add_spatial = self.ray_from_mask(m) * bg
        colored_out = colored + add_spatial * present
        return torch.cat([bg * (1.0 - add_spatial), colored_out], dim=1)


class Task051CentroidRay(CoordModule):
    """Single marker colour in an arrow-like object: extend the marker ray toward the main object centroid."""
    def __init__(self):
        super().__init__()

    def forward(self, x):
        bg = x[:, 0:1]
        colored = x[:, 1:CH]
        counts = colored.sum(dim=(2, 3), keepdim=True)
        present = (counts > 0.0).float()
        adjusted = torch.where(present > 0.0, counts, torch.ones_like(counts) * self.big)
        min_count = torch.amin(adjusted, dim=1, keepdim=True)
        marker_selector = ((counts - min_count).abs() < 0.5).float() * present
        marker_mask = (colored * marker_selector).sum(dim=1, keepdim=True)
        main_mask = colored.sum(dim=1, keepdim=True) - marker_mask
        marker_count = marker_mask.sum(dim=(2, 3), keepdim=True).clamp(min=1.0)
        main_count = main_mask.sum(dim=(2, 3), keepdim=True).clamp(min=1.0)
        mr = (marker_mask * self.rr).sum(dim=(2, 3), keepdim=True) / marker_count
        mc = (marker_mask * self.cc).sum(dim=(2, 3), keepdim=True) / marker_count
        cr = (main_mask * self.rr).sum(dim=(2, 3), keepdim=True) / main_count
        cc = (main_mask * self.cc).sum(dim=(2, 3), keepdim=True) / main_count
        dr = cr - mr
        dc = cc - mc
        vertical = (dr.abs() > dc.abs()).float()
        horizontal = 1.0 - vertical
        row_match = ((self.rr - mr).abs() < 0.5).float()
        col_match = ((self.cc - mc).abs() < 0.5).float()
        ray_right = horizontal * (dc > 0.0).float() * row_match * (self.cc > mc).float()
        ray_left  = horizontal * (dc < 0.0).float() * row_match * (self.cc < mc).float()
        ray_down  = vertical   * (dr > 0.0).float() * col_match * (self.rr > mr).float()
        ray_up    = vertical   * (dr < 0.0).float() * col_match * (self.rr < mr).float()
        add_spatial = ((ray_right + ray_left + ray_down + ray_up) > 0.0).float() * bg
        colored_out = colored + add_spatial * marker_selector
        return torch.cat([bg * (1.0 - add_spatial), colored_out], dim=1)


class Task378FrameCornerRays(CoordModule):
    """Dense payload rectangle inside a sparse frame: extend payload-colour diagonals from the far closed frame corner(s)."""
    def __init__(self):
        super().__init__()

    def forward(self, x):
        bg = x[:, 0:1]
        colored = x[:, 1:CH]
        counts = colored.sum(dim=(2, 3), keepdim=True)
        present = (counts > 0.0).float()
        rr = self.rr
        cc = self.cc
        big = torch.ones_like(counts) * self.big
        small = torch.ones_like(counts) * (-self.big)
        rmin = torch.amin(torch.where(colored > 0.0, rr, torch.ones_like(colored) * self.big), dim=(2, 3), keepdim=True)
        rmax = torch.amax(torch.where(colored > 0.0, rr, torch.ones_like(colored) * (-self.big)), dim=(2, 3), keepdim=True)
        cmin = torch.amin(torch.where(colored > 0.0, cc, torch.ones_like(colored) * self.big), dim=(2, 3), keepdim=True)
        cmax = torch.amax(torch.where(colored > 0.0, cc, torch.ones_like(colored) * (-self.big)), dim=(2, 3), keepdim=True)
        height = (rmax - rmin + 1.0).clamp(min=1.0)
        width = (cmax - cmin + 1.0).clamp(min=1.0)
        area = height * width
        density = torch.where(present > 0.0, counts / area, torch.zeros_like(counts) - 1.0)
        max_density = torch.amax(density, dim=1, keepdim=True)
        marker_selector = ((density - max_density).abs() < 1.0e-5).float() * present
        frame_selector = present * (1.0 - marker_selector)
        marker_mask = (colored * marker_selector).sum(dim=1, keepdim=True)
        frame_mask = (colored * frame_selector).sum(dim=1, keepdim=True)
        rfmin = (rmin * frame_selector).sum(dim=1, keepdim=True)
        rfmax = (rmax * frame_selector).sum(dim=1, keepdim=True)
        cfmin = (cmin * frame_selector).sum(dim=1, keepdim=True)
        cfmax = (cmax * frame_selector).sum(dim=1, keepdim=True)
        fheight = (rfmax - rfmin + 1.0).clamp(min=1.0)
        fwidth = (cfmax - cfmin + 1.0).clamp(min=1.0)
        mcount = marker_mask.sum(dim=(2, 3), keepdim=True).clamp(min=1.0)
        mr = (marker_mask * rr).sum(dim=(2, 3), keepdim=True) / mcount
        mc = (marker_mask * cc).sum(dim=(2, 3), keepdim=True) / mcount
        rcenter = (rfmin + rfmax) * 0.5
        ccenter = (cfmin + cfmax) * 0.5
        far_top = (mr >= rcenter).float()
        far_bottom = (mr <= rcenter).float()
        far_left = (mc >= ccenter).float()
        far_right = (mc <= ccenter).float()
        row_top = ((rr - rfmin).abs() < 0.5).float()
        row_bottom = ((rr - rfmax).abs() < 0.5).float()
        col_left = ((cc - cfmin).abs() < 0.5).float()
        col_right = ((cc - cfmax).abs() < 0.5).float()
        top_full = (frame_mask * row_top).sum(dim=(2, 3), keepdim=True) >= (fwidth - 0.5)
        bottom_full = (frame_mask * row_bottom).sum(dim=(2, 3), keepdim=True) >= (fwidth - 0.5)
        left_full = (frame_mask * col_left).sum(dim=(2, 3), keepdim=True) >= (fheight - 0.5)
        right_full = (frame_mask * col_right).sum(dim=(2, 3), keepdim=True) >= (fheight - 0.5)
        top_full = top_full.float(); bottom_full = bottom_full.float(); left_full = left_full.float(); right_full = right_full.float()
        # Four possible far closed corners. Each ray excludes the frame corner itself.
        tl_ok = far_top * far_left * top_full * left_full
        tr_ok = far_top * far_right * top_full * right_full
        bl_ok = far_bottom * far_left * bottom_full * left_full
        br_ok = far_bottom * far_right * bottom_full * right_full
        ray_tl = tl_ok * (rr < rfmin).float() * (cc < cfmin).float() * (((rfmin - rr) - (cfmin - cc)).abs() < 0.5).float()
        ray_tr = tr_ok * (rr < rfmin).float() * (cc > cfmax).float() * (((rfmin - rr) - (cc - cfmax)).abs() < 0.5).float()
        ray_bl = bl_ok * (rr > rfmax).float() * (cc < cfmin).float() * (((rr - rfmax) - (cfmin - cc)).abs() < 0.5).float()
        ray_br = br_ok * (rr > rfmax).float() * (cc > cfmax).float() * (((rr - rfmax) - (cc - cfmax)).abs() < 0.5).float()
        add_spatial = ((ray_tl + ray_tr + ray_bl + ray_br) > 0.0).float() * bg
        colored_out = colored + add_spatial * marker_selector
        return torch.cat([bg * (1.0 - add_spatial), colored_out], dim=1)


def make_symbolic_model(task_id):
    if task_id == 'task051':
        return Task051CentroidRay()
    if task_id == 'task168':
        return Task168DiagonalLCompletion()
    if task_id == 'task378':
        return Task378FrameCornerRays()
    raise ValueError(task_id)


def export_symbolic_model(task_id, out_path):
    model = make_symbolic_model(task_id).eval()
    dummy = torch.zeros(1, CH, H, W, dtype=torch.float32)
    torch.onnx.export(model, dummy, str(out_path), input_names=['input'], output_names=['output'],
                      opset_version=17, dynamic_axes=None, do_constant_folding=True, dynamo=False)
    onnx_model = onnx.load(str(out_path))
    onnx_model.ir_version = 8
    onnx.checker.check_model(onnx_model)
    onnx.save(onnx_model, str(out_path))


def build_model(task, out_path, config):
    export_symbolic_model(config['task_id'], out_path)
    return {'builder': config['builder'], 'strategy': config['note']}


def run_onnx_grid(sess, grid):
    out = sess.run(['output'], {'input': grid_to_tensor(grid)})[0]
    return (out > 0.5).astype(np.float32)


def validate_model(model_path, task, scope='visible'):
    sess = ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
    examples = examples_for_scope(task, scope)
    right = 0; total = 0; first_wrong = None
    for i, ex in enumerate(examples):
        total += 1
        ok = np.array_equal(run_onnx_grid(sess, ex['input']), expected_tensor(ex['output']))
        if ok:
            right += 1
        elif first_wrong is None:
            first_wrong = i
    model = onnx.load(str(model_path))
    ops = {}
    for node in model.graph.node:
        ops[node.op_type] = ops.get(node.op_type, 0) + 1
    bad = sorted(FORBIDDEN & set(ops))
    return {'right': right, 'total': total, 'first_wrong': first_wrong,
            'file_size_bytes': Path(model_path).stat().st_size,
            'under_1_4mb': Path(model_path).stat().st_size < 1_400_000,
            'forbidden_ops_present': bad, 'op_counts': ops}


In [4]:
from pathlib import Path
import json

ROOT = Path.cwd()
TASK_CONFIG = {'task_id': TASK_ID, 'builder': 'symbolic_task378_frame_corner_rays', 'train_scope': 'none', 'verify_scope': 'all', 'note': 'separate dense payload colour from sparse frame colour, choose far closed frame corner(s) relative to the payload centroid, and emit payload-colour diagonal rays'}
DATA_PATH = find_task_json(TASK_ID)
OUT_DIR = ROOT / f'working_submission_{TASK_ID}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUT_DIR / f'{TASK_ID}.onnx'
print('DATA_PATH =', DATA_PATH)
print('OUT_DIR =', OUT_DIR)
print('MODEL_PATH =', MODEL_PATH)
print('TASK_CONFIG =', TASK_CONFIG)

DATA_PATH = /kaggle/input/competitions/neurogolf-2026/task378.json
OUT_DIR = /kaggle/working/working_submission_task378
MODEL_PATH = /kaggle/working/working_submission_task378/task378.onnx
TASK_CONFIG = {'task_id': 'task378', 'builder': 'symbolic_task378_frame_corner_rays', 'train_scope': 'none', 'verify_scope': 'all', 'note': 'separate dense payload colour from sparse frame colour, choose far closed frame corner(s) relative to the payload centroid, and emit payload-colour diagonal rays'}


In [5]:
task, task_path = load_task(TASK_ID)
print('task path:', task_path)
print('train examples:', len(task.get('train', [])))
print('test examples:', len(task.get('test', [])))
print('arc-gen examples:', len(task.get('arc-gen', [])))
print('first input shape:', (len(task['train'][0]['input']), len(task['train'][0]['input'][0])))
print('first output shape:', (len(task['train'][0]['output']), len(task['train'][0]['output'][0])))

task path: /kaggle/input/competitions/neurogolf-2026/task378.json
train examples: 4
test examples: 1
arc-gen examples: 262
first input shape: (6, 6)
first output shape: (6, 6)


In [6]:
# Small inspection: color counts for the first training pair.
from collections import Counter
import numpy as np
first = task['train'][0]
print('input colors:', Counter(np.array(first['input']).ravel()))
print('output colors:', Counter(np.array(first['output']).ravel()))
print('changed cells:', int(np.sum(np.array(first['input']) != np.array(first['output']))))

input colors: Counter({np.int64(0): 25, np.int64(9): 7, np.int64(3): 4})
output colors: Counter({np.int64(0): 23, np.int64(9): 7, np.int64(3): 6})
changed cells: 2


In [7]:
# Export plan for this task.
plan = {
    'task_id': TASK_ID,
    'builder': TASK_CONFIG['builder'],
    'extractor': TASK_CONFIG.get('extractor'),
    'max_depth': TASK_CONFIG.get('max_depth'),
    'train_scope': TASK_CONFIG['train_scope'],
    'verify_scope': TASK_CONFIG['verify_scope'],
    'model_path': str(MODEL_PATH),
}
plan

{'task_id': 'task378',
 'builder': 'symbolic_task378_frame_corner_rays',
 'extractor': None,
 'max_depth': None,
 'train_scope': 'none',
 'verify_scope': 'all',
 'model_path': '/kaggle/working/working_submission_task378/task378.onnx'}

In [8]:
# Task-specific model construction wrapper.
def build_current_model():
    task, _ = load_task(TASK_ID)
    return build_model(task, MODEL_PATH, TASK_CONFIG)

def validate_current_model(scope=None):
    task, _ = load_task(TASK_ID)
    return validate_model(MODEL_PATH, task, scope or TASK_CONFIG['verify_scope'])

In [9]:
# Build ONNX model and enforce competition constraints.
build_info = build_current_model()
validation_report = validate_current_model()
assert validation_report['right'] == validation_report['total'], validation_report
assert validation_report['under_1_4mb'], validation_report['file_size_bytes']
assert not validation_report['forbidden_ops_present'], validation_report['forbidden_ops_present']
print('build_info:', build_info)
print('validation_report:', validation_report)

/tmp/ipykernel_16/1486363708.py:248: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, str(out_path), input_names=['input'], output_names=['output'],


build_info: {'builder': 'symbolic_task378_frame_corner_rays', 'strategy': 'separate dense payload colour from sparse frame colour, choose far closed frame corner(s) relative to the payload centroid, and emit payload-colour diagonal rays'}
validation_report: {'right': 267, 'total': 267, 'first_wrong': None, 'file_size_bytes': 152652, 'under_1_4mb': True, 'forbidden_ops_present': [], 'op_counts': {'Constant': 43, 'Slice': 2, 'ReduceSum': 14, 'Greater': 6, 'Cast': 23, 'Where': 5, 'ReduceMin': 2, 'ReduceMax': 3, 'Sub': 19, 'Add': 10, 'Clip': 5, 'Mul': 44, 'Div': 3, 'Abs': 9, 'Less': 11, 'GreaterOrEqual': 6, 'LessOrEqual': 2, 'Concat': 1}}


In [10]:
# Optional broader arc-gen check. This is asserted only when verify_scope == 'all'.
optional_all_report = validate_model(MODEL_PATH, task, 'all')
print('optional_all_report:', optional_all_report)
if TASK_CONFIG['verify_scope'] == 'all':
    assert optional_all_report['right'] == optional_all_report['total'], optional_all_report

optional_all_report: {'right': 267, 'total': 267, 'first_wrong': None, 'file_size_bytes': 152652, 'under_1_4mb': True, 'forbidden_ops_present': [], 'op_counts': {'Constant': 43, 'Slice': 2, 'ReduceSum': 14, 'Greater': 6, 'Cast': 23, 'Where': 5, 'ReduceMin': 2, 'ReduceMax': 3, 'Sub': 19, 'Add': 10, 'Clip': 5, 'Mul': 44, 'Div': 3, 'Abs': 9, 'Less': 11, 'GreaterOrEqual': 6, 'LessOrEqual': 2, 'Concat': 1}}


In [11]:
# Model/version manifest.
run_manifest = {
    'task_id': TASK_ID,
    'model_version': MODEL_VERSION,
    'strategy': TASK_CONFIG['note'],
    'build_info': build_info,
    'validation_report': validation_report,
    'optional_all_report': optional_all_report,
}
run_manifest

{'task_id': 'task378',
 'model_version': 'task378-symbolic-frame-corner-rays-v1',
 'strategy': 'separate dense payload colour from sparse frame colour, choose far closed frame corner(s) relative to the payload centroid, and emit payload-colour diagonal rays',
 'build_info': {'builder': 'symbolic_task378_frame_corner_rays',
  'strategy': 'separate dense payload colour from sparse frame colour, choose far closed frame corner(s) relative to the payload centroid, and emit payload-colour diagonal rays'},
 'validation_report': {'right': 267,
  'total': 267,
  'first_wrong': None,
  'file_size_bytes': 152652,
  'under_1_4mb': True,
  'forbidden_ops_present': [],
  'op_counts': {'Constant': 43,
   'Slice': 2,
   'ReduceSum': 14,
   'Greater': 6,
   'Cast': 23,
   'Where': 5,
   'ReduceMin': 2,
   'ReduceMax': 3,
   'Sub': 19,
   'Add': 10,
   'Clip': 5,
   'Mul': 44,
   'Div': 3,
   'Abs': 9,
   'Less': 11,
   'GreaterOrEqual': 6,
   'LessOrEqual': 2,
   'Concat': 1}},
 'optional_all_report': 

In [12]:
# Architecture report.
model = onnx.load(str(MODEL_PATH))
op_counts = {}
for node in model.graph.node:
    op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1
architecture_report = {
    'file_size_bytes': MODEL_PATH.stat().st_size,
    'nodes': len(model.graph.node),
    'op_counts': op_counts,
    'forbidden_ops_present': sorted(FORBIDDEN & set(op_counts)),
}
architecture_report

{'file_size_bytes': 152652,
 'nodes': 208,
 'op_counts': {'Constant': 43,
  'Slice': 2,
  'ReduceSum': 14,
  'Greater': 6,
  'Cast': 23,
  'Where': 5,
  'ReduceMin': 2,
  'ReduceMax': 3,
  'Sub': 19,
  'Add': 10,
  'Clip': 5,
  'Mul': 44,
  'Div': 3,
  'Abs': 9,
  'Less': 11,
  'GreaterOrEqual': 6,
  'LessOrEqual': 2,
  'Concat': 1},
 'forbidden_ops_present': []}

In [13]:
# Persist metadata next to ONNX.
manifest_path = OUT_DIR / f'{TASK_ID}_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('wrote model:', MODEL_PATH)
print('wrote manifest:', manifest_path)

wrote model: /kaggle/working/working_submission_task378/task378.onnx
wrote manifest: /kaggle/working/working_submission_task378/task378_manifest.json


In [14]:
# Package single-task submission zip.
zip_path = ROOT / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(MODEL_PATH, MODEL_PATH.name)
print('wrote zip:', zip_path)

wrote zip: /kaggle/working/submission.zip


In [15]:
# Persist run metadata next to the generated models.
if 'profile_df' in globals() and len(profile_df):
    profile_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_profile.csv'
    profile_df.to_csv(profile_path, index=False)
    print('wrote profile:', profile_path)

manifest_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('wrote manifest:', manifest_path)

wrote manifest: /kaggle/working/working_submission_task378/arc_static_symbolic_task378-symbolic-frame-corner-rays-v1_manifest.json
